In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

from scipy import stats

In [ ]:
df = pd.read_csv("churn.csv")


In [ ]:
print(df.head())
print(df.info())
print(df.describe())
# Target distribution
print(df['Churn'].value_counts())


In [ ]:

df = df.dropna()

In [ ]:

le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

# ---- One-Hot Encoding ----
df = pd.get_dummies(df, columns=['InternetService'], drop_first=True)

# ---- Frequency Encoding ----
freq = df['Contract'].value_counts()
df['Contract_freq'] = df['Contract'].map(freq)

# Convert target to numeric if needed
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

In [ ]:

# Standard Scaling
scaler = StandardScaler()
df['MonthlyCharges_scaled'] = scaler.fit_transform(df[['MonthlyCharges']])

# Min-Max Scaling
mm = MinMaxScaler()
df['tenure_scaled'] = mm.fit_transform(df[['tenure']])

In [ ]:


# ---- IQR Method ----
Q1 = df['MonthlyCharges'].quantile(0.25)
Q3 = df['MonthlyCharges'].quantile(0.75)
IQR = Q3 - Q1

df = df[(df['MonthlyCharges'] >= Q1 - 1.5 * IQR) &
        (df['MonthlyCharges'] <= Q3 + 1.5 * IQR)]

# ---- Z-Score Method ----
z = np.abs(stats.zscore(df['tenure']))
df = df[z < 3]

In [ ]:

# 1. Total Spend
df['TotalSpend'] = df['MonthlyCharges'] * df['tenure']

# 2. Average Value
df['AvgValue'] = df['TotalSpend'] / (df['tenure'] + 1)

# 3. Tenure Group
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0,12,24,48,72], labels=[1,2,3,4])

# Convert category to numeric
df['TenureGroup'] = df['TenureGroup'].astype(int)

# 4. Charges Per Service (assuming NumServices exists)
if 'NumServices' in df.columns:
    df['ChargesPerService'] = df['MonthlyCharges'] / (df['NumServices'] + 1)
else:
    df['ChargesPerService'] = df['MonthlyCharges'] / 2

# 5. Payment Efficiency
df['PaymentEfficiency'] = df['TotalSpend'] / (df['MonthlyCharges'] + 1)

# 6. Contract Score
df['ContractScore'] = df['Contract_freq'] * df['tenure']


In [ ]:


# Correlation Heatmap
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

# Feature Importance
X = df.drop('Churn', axis=1)
y = df['Churn']

model = RandomForestClassifier()
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=X.columns)
print("\nFeature Importance:\n", importance.sort_values(ascending=False))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import pickle

pickle.dump(pipeline, open("churn_model.pkl", "wb"))

print("\nModel saved successfully!")